# Lesson 50 · The full MFCC pipeline — understanding a frame of sound in 13 numbers (STFT → mel → log → DCT)

**Goal**: finally wire the previous lessons together — STFT → mel → log → DCT — implement the complete MFCC by hand, and match `mfcc()` numerically.

> **The heart of the features**: the usual choice is the first **13** coefficients (engineering experience: the low-order cepstrum is already enough to tell timbres apart; higher orders are mostly noise). This also links back to L09 (vectors) and L11 (normalization).

🔗 Aurora connection: `aurora.audio.mfcc.mfcc()`, `aurora.audio.mfcc.dct_ii()`, `aurora.audio.mel.mel_spectrogram()`; these features are the standard input to the Month 2 keyword-spotting classifier.

← **Previous lesson** · [L49 · The DCT-II](L49_dct.ipynb)

> Last lesson covered **the DCT-II**: why it decorrelates, and a pure-NumPy replacement for scipy.fft.dct.
> This lesson explores **the full MFCC pipeline**.

## This lesson's story: how does a phone's voice assistant recognize your voice from 13 numbers?

You say one syllable, and the phone's microphone records about 0.03 seconds (30 ms) of sound — roughly 480 samples. Those 480 numbers are raw PCM, full of redundancy and noise.

What a speech-recognition model actually takes in is not those 480 numbers but **13 MFCC coefficients** — a 5-step pipeline squeezes the redundancy out:

```
x      (T,)                    ← raw audio
  ↓ STFT + power spectrum
P      (n_frames, n_fft//2+1)  ← power spectrum per frame (phase dropped)
  ↓ × mel_filterbank
M      (n_frames, n_mels)      ← mel energies (mimicking the ear)
  ↓ log(·+ε)
logM   (n_frames, n_mels)      ← log compression (mimicking loudness perception)
  ↓ DCT-II
MFCC   (n_frames, n_mfcc)      ← decorrelated cepstral coefficients ✓
```

**What each step means physically**:

| Step | Operation | What it removes | What it keeps |
|---|---|---|---|
| STFT | framing + FFT | waveform detail in the time domain | frequency content |
| Mel | triangular filters | high-frequency redundancy | frequencies as the ear perceives them |
| log | take the logarithm | absolute loudness | relative loudness differences |
| DCT | cosine transform | correlation between channels | independent spectral-envelope parameters |

This section's task: implement `my_mfcc(x, sr, n_mfcc=13, n_mels=40, win_len=1024, hop=256)` with an error < 1e-5 against `aurora.audio.mfcc.mfcc`.

## Why can "mel frequency" mimic the human ear?

Before the mel filters, let's look at how the ear itself works.

Inside your ear is a spiral-shaped structure called the **cochlea**. When sound comes in, different frequencies vibrate most strongly at different places along the cochlea — **high frequencies near the entrance, low frequencies further in**. This is called **frequency decomposition**.

Even more important: **the ear does not perceive frequency linearly**.
- A change from 100 Hz to 200 Hz we hear as "an obvious change in pitch"
- A change from 5000 Hz to 5100 Hz we can barely hear at all

In other words: **the ear has high resolution at low frequencies and low resolution at high frequencies**.

In psychoacoustics, Stevens, Volkmann and Newman ran a large set of listening experiments in 1937 and drew the curve relating "physical frequency" to "what the ear perceives"; that curve was later written down as the mel frequency scale. Reorganizing the spectrum on this scale means:
- more filters at low frequencies (25 Hz spacing)
- fewer filters at high frequencies (100+ Hz spacing)

That gives 40 mel filters that are "evenly spaced psychologically" — matching how the ear really perceives frequency.

A side-by-side comparison:
- Linear frequency: 0, 50, 100, 150, ..., 8000 Hz (equal spacing) → too dense at high frequencies, too sparse at low ones
- Mel frequency: roughly 0, 20, 40, 65, 110, ..., 8000 Hz (dense at low frequencies, sparse at high ones) → matches perception

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import sine

## 1. The whole formula at a glance

```
MFCC[n] = DCT-II( log( mel_filterbank @ |STFT|² ) )[:n_mfcc]
```

- `|STFT|²` is each frame's **power spectrum**, shape `(n_frames, n_fft//2+1)`
- `mel_filterbank @ |STFT|²` maps the power spectrum onto mel bands, shape `(n_frames, n_mels)`
- `log(...)` compresses the dynamic range (a 100× difference in power → only 20 apart in dB)
- `DCT-II` decorrelates; coefficients from low to high carry spectral-envelope information from coarse to fine
- keep the first `n_mfcc` coefficients (`[:n_mfcc]`); k=0 is the DC mean of all the mel energies, and keeping it matches librosa's default behavior

For the orthonormal (`norm="ortho"`) DCT-II, the `k`-th coefficient is:

```
y[k] = w[k] * sum_{n=0}^{N-1} x[n] * cos( pi/N * (n + 0.5) * k )
```

where `w[0] = sqrt(1/N)` and `w[k>0] = sqrt(2/N)`.

### 📌 Common questions cleared up

**❓ Question 1: which way round is the mel_filterbank matrix product?**

It looks like it should be `mel_filterbank @ |STFT|²`, but the code actually says `power @ fb.T`. Why are these equivalent?

**Answer**: it comes down to the **row/column convention** of the matrices.
- `mel_filterbank` (call it `fb`) has shape `(n_mels, n_fft//2+1)` — each row is one mel filter
- `power` has shape `(n_frames, n_fft//2+1)` — each row is one frame's power spectrum

**Mathematically**, we want to dot "the power spectrum of frame i" with "mel filter j" to get "the j-th mel energy of frame i".
- With `fb @ power.T` → `(n_mels, n_fft//2+1) @ (n_fft//2+1, n_frames)` = `(n_mels, n_frames)`, which you then have to transpose
- With `power @ fb.T` → `(n_frames, n_fft//2+1) @ (n_fft//2+1, n_mels)` = `(n_frames, n_mels)` ✓ the target shape directly

So the code uses `power @ fb.T` so that rows of the result are frames and columns are mel channels. Remember this convention and you won't mix them up.

**❓ Question 2: why is power_to_db 10·log10 rather than ln?**

We want to compress the dynamic range. Say two powers differ by 100×:
- `ln(100)` ≈ 4.6
- `log10(100) = 2`

The two differ only by a constant factor (ln = 2.303·log10); changing the base doesn't make the compression any more "aggressive".

The extra factor of **10** comes from **the definition of the decibel (dB)**: for power quantities, decibels = 10·log10(P/P_ref). That's an engineering convention, not a law of nature — a decibel is defined that way, just as a meter is a meter.

So `10·log10` isn't an arbitrary choice; it puts the result in the industry-standard unit, the decibel — use ln and your numbers won't line up with the dB values in the engineering literature.

**❓ Question 2, deeper: why do speech-recognition models work better once we "compress to dB"?**

This is where Samuel and Hannah both kept digging — the maths `log10(100) = 2` is easy, but what is the actual cause-and-effect between "the numbers get compressed" and "recognition gets better"? Knowing the formula isn't enough; first we need to see "why it has to be a logarithm, and not just dividing by some constant to make the numbers smaller".

Start with an everyday scene: you're in a quiet library and someone next to you gently turns a page — "that's so clear", you think. But stand next to the speakers at a concert, and the same page-turn with the same force is completely inaudible. The same physical intensity, two settings, and the loudness you *perceive* is worlds apart. Your ear isn't measuring "how many joules of energy this sound has"; it's comparing "how many times louder this sound is than the current background".

This isn't an illusion; it's a real physiological law, known in psychology as **Weber's law**: how strongly we perceive a change in a physical stimulus is proportional to **the size of the change divided by the current baseline level of the stimulus**, not to the change itself:

```
perceived change in loudness  ∝  ΔP / P        (P is the physical intensity of the sound, e.g. power)
```

If you "add up" this relationship in the language of calculus (stack up infinitely many tiny proportional changes), you naturally get that perceived loudness is proportional to `log(P)`. That's the root reason the decibel was built on a log scale from the start — not to make the numbers look smaller, but because **it uses maths to reproduce what the ear is already doing**.

So what does this have to do with speech-recognition models? Two angles:

1. **A representation that agrees with the ear makes the model's "numerical differences" closer to "perceptual differences"**. The same vowel "ah", said softly or shouted, can differ by 100× or even 1000× in the power spectrum. Without the log, the model sees inputs that are numerically miles apart and is easily misled into "these are two completely different signals" — yet to the ear, and as far as the words go, it's plainly "the same phoneme, just at a different loudness". After the log, a 100× power difference shrinks to a numerical difference of 20 (dB), and the effect of volume on the feature values is greatly weakened.
2. **It stops loud signals from numerically "drowning out" quiet ones**. What MFCC really wants to keep is the **relative shape** of the spectrum — which frequency components are strong, which are weak, their ratios to each other; that's the key information deciding "which sound this is". With raw linear power values, the loud frames have huge numbers and the quiet frames tiny ones, and during training the model drifts into focusing all its attention on "loudness", a dimension that has little to do with content. After the log, the differences in relative shape across loudness levels are preserved and the effect of absolute loudness is compressed, so the model finds it much easier to learn "shape" rather than "volume".

One-sentence summary: **taking the log isn't about making numbers small and comfortable to look at; it turns "multiplicative relationships in physical intensity" into "additive relationships in the numbers" — which happens to be how the ear perceives loudness, and it stops the "spectral shape" information that really matters from being drowned out by the "volume" information that doesn't.**

**❓ Question 3: how does the 1e-10 floor protection work?**

The power spectrum contains values close to 0 (especially where high-frequency noise is tiny). Taking `log(0)` directly → -∞, which produces nan or inf.

What we actually do is `log(max(power, 1e-10))`: **any value below 1e-10 is treated as 1e-10 before taking the log**. That way, even if the input is all zeros, the output is a finite negative number (about -100 dB) rather than -∞.

Could you equivalently write `log(power + 1e-10)`? **No** — the first handles "the value is too small", the second "adds a bit of noise", and the two treat small values differently. `aurora.audio.power_to_db` uses the `max(power, amin)` form.

**❓ Question 4: why n_frames = 1 + L // hop (full derivation)**

This comes from the STFT's `center=True` mechanism. Let's work out, step by step, where "that extra frame" comes from.

**Step 1: figure out how much padding is added.**
With `center=True`, Aurora mirror-pads (reflects) `n_fft // 2` samples at each end of the signal, so the first frame's window is centered on sample 0 of the original signal and the last frame's window is centered on its last sample. After padding the total length becomes:
```
L' = L + 2 * (n_fft // 2) = L + n_fft   (exact when n_fft is even; this lesson's n_fft=1024 qualifies)
```

**Step 2: count how many windows "of length n_fft, spaced hop apart" fit in a signal of length L'.**
This is a "fence-post counting" problem: if each window takes n_fft samples and consecutive window starts are hop apart, the last window can start at most at `L' - n_fft` (any later and the window runs off the end of the signal). Window starts go `0, hop, 2·hop, ...`, so the number of windows is:
```
n_frames = floor( (L' - n_fft) / hop ) + 1     ← the "+1" is because start 0 counts as a frame too
```

**Step 3: substitute L' = L + n_fft, and notice that n_fft cancels exactly — this step is the key one.**
```
L' - n_fft = (L + n_fft) - n_fft = L        ← exact cancellation, not an approximation
n_frames  = floor(L / hop) + 1 = 1 + L // hop
```

**This is an exact formula, not an estimate that "only holds approximately when L is much larger than n_fft"** — if you divide `L' = L + n_fft` by hop directly and forget to subtract n_fft first, you get a wrong, too-large frame count. Checking both methods with this lesson's parameters:

```
L = 16000, n_fft = 1024, hop = 256

❌ Wrong method (divide L' by hop directly, forgetting to subtract n_fft):
   (L + n_fft) // hop + 1 = 17024 // 256 + 1 = 66 + 1 = 67     ← wrong, 4 frames too many

✅ Right method (subtract n_fft first, then divide by hop):
   (L' - n_fft) // hop + 1 = 16000 // 256 + 1 = 62 + 1 = 63    ← right
   equivalent to 1 + L // hop = 1 + 62 = 63                     ← matches the closed-book check cell and the code demo
```

With `center=False` (no mirror padding) there's no "add n_fft, then subtract n_fft" cancellation, and the frame count depends directly on the signal length and window length: `n_frames = (L - n_fft) // hop + 1` (requires `L ≥ n_fft`). Aurora's STFT defaults to `center=True`, so this lesson uses `1 + L // hop` throughout.

The full `center=True` mirror-padding mechanism (why the padding is symmetric, exactly how boundary samples are reflected) is illustrated and derived in **L44 (the STFT lesson)**; if it still hasn't fully clicked, going back through that lesson will make it solid.


In [ ]:
# Demo: the decorrelating effect of the DCT-II
from aurora.audio.mfcc import dct_ii

sr = 16000
x = sine(440, duration=0.5, sample_rate=sr)

# Build one frame of log-mel spectrum (40 mel bins)
from aurora.audio.mel import mel_spectrogram, power_to_db
log_mel = power_to_db(mel_spectrogram(x, sr, n_mels=40, n_fft=1024, hop_length=256), top_db=None)
# Take frame 5 as an example
frame = log_mel[5]
dct_frame = dct_ii(frame)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar(range(len(frame)), frame)
axes[0].set_title("log-mel spectrum (40 bins, frame 5)")
axes[0].set_xlabel("mel bin")
axes[0].set_ylabel("dB")
axes[1].bar(range(len(dct_frame)), dct_frame, color="orange")
axes[1].axvline(12.5, color="red", linestyle="--", label="cutoff: keep k=0..12, n_mfcc=13 in all")
axes[1].set_title("DCT-II coefficients (keep k=0..12; n_mfcc=13 includes k=0)")
axes[1].set_xlabel("DCT coefficient index k")
axes[1].legend()
plt.tight_layout()
plt.show()

# Print the actual values
print("📊 How the DCT coefficients' energy is distributed:")
print(f"k=0 (DC)  : {dct_frame[0]:.4f}  ← largest, represents the overall energy")
print(f"k=1..4    : {dct_frame[1:5].round(4)}  ← falling off fast")
print(f"k=5..8    : {dct_frame[5:9].round(4)}  ← still falling")
print(f"k=9..12   : {dct_frame[9:13].round(4)}  ← close to zero")
print(f"k=13..19  : {dct_frame[13:20].round(4)}  ← high-order coefficients are tiny (noise)")
print(f"\n💡 Keeping the first 13 coefficients already captures most of the information; the high-order coefficients carry very little energy.")

## 2. Why is n_mfcc usually 13?

The spectral envelope of speech can be described with very few parameters. After the DCT, **the energy is concentrated in the low-order coefficients**, and the high-order ones are close to zero. Why?

### What the DCT coefficients mean physically

Picture a smooth curve (the mel power spectrum) next to a very jittery one.
- The DCT's low-order coefficients (k=0, 1, 2, ...) **capture large-scale shape**: overall energy, the general envelope, slow peaks and valleys
- The DCT's high-order coefficients (k=30, 31, ..., 39) **capture small-scale detail**: spikes, noise, fast fluctuations

For the **human voice**, this split is very useful:
- Low-order coefficients (k=0–5) mostly reflect **the fixed shape of the vocal tract** — the geometry of your mouth, throat and nasal cavity. This determines your timbre (the character of vowels)
  - For example, when anyone says "ah", male or female voice, the large-scale features of the vocal-tract shape are similar
- Mid-order coefficients (k=6–12) reflect **articulation detail**: lip shape, airflow strength and so on
- High-order coefficients (k=13+) reflect **noise and artifacts**: microphone noise, mouth noises, background sounds

Experiments show that the first 13 coefficients (k=0–12) already contain most of the "speech-feature" information. The higher coefficients are essentially "noise from the environment or from speaker characteristics".

| n_mfcc | Typical use |
|--------|---------|
| 13 | classic MFCC, the standard configuration for GMM-HMM speech recognition (proven in industry for 30+ years) |
| 20–40 | deep-learning classifiers; more information (a neural network can learn to ignore the noise) |
| > 40 | diminishing returns; equivalent to using log-mel directly (losing the DCT's decorrelation advantage) |

`n_mels` controls the resolution of the mel filterbank (40–128 is common); `n_mfcc` ≤ `n_mels`. Raising `n_mfcc` means keeping more high-order spectral detail, at the cost of more overfitting risk and computation.

### Why do k=0 and k>0 get different DCT weights?

We said earlier that the DCT-II's orthonormal weights are `w[0] = sqrt(1/N)` and `w[k>0] = sqrt(2/N)`. Why?

In short: **the DCT's 0th basis vector (all ones) is different from the others**.
- Basis 0 = [1, 1, 1, ..., 1]  → represents the "DC component" (the average of all the signal's elements)
- The other bases = cosine waves → represent the "AC components"

For all these bases to have unit length in the L2-norm sense (orthonormalization), the DC component's weight has to be `sqrt(1/N)` rather than `sqrt(2/N)`. That's standard linear-algebra practice and tied to how the discrete cosine transform itself is defined.

**Remember this**: when you see "the weights are defined differently", there's usually some difference in symmetry or physical meaning behind it. Here, the difference is "DC vs AC".

### Working it out from scratch: the actual numbers for N=4

That paragraph may still feel a bit abstract — words like "L2 norm" and "unit length" are hard to get a feel for without computing anything. Let's plug in the smallest example you can do by hand — **N=4** — and work out every number. You'll see the weight formula isn't decreed out of thin air; it's the inevitable result of "working backward".

The n-th component of the DCT-II's k-th basis vector is:
```
e_k[n] = cos( π/N · (n + 0.5) · k )
```

**First, the 0th basis vector (k=0)**: since `cos(0) = 1`, `e_0[n]` is always 1 whatever n is. So for N=4:
```
e_0 = [1, 1, 1, 1]
```
How do we compute this vector's "length" (L2 norm)? Square each component, add them up, take the square root — Pythagoras extended to four dimensions:
```
‖e_0‖ = sqrt(1² + 1² + 1² + 1²) = sqrt(4) = 2 = sqrt(N)      (N=4, so sqrt(N)=2)
```

**Next, the 1st basis vector (k=1, N=4)**: plug in n=0,1,2,3 and all four numbers can be worked out by hand (to 4 decimal places):
```
e_1[0] = cos(π/4 · 0.5 · 1) = cos(22.5°) ≈ 0.9239
e_1[1] = cos(π/4 · 1.5 · 1) = cos(67.5°) ≈ 0.3827
e_1[2] = cos(π/4 · 2.5 · 1) = cos(112.5°) ≈ -0.3827
e_1[3] = cos(π/4 · 3.5 · 1) = cos(157.5°) ≈ -0.9239
```
Its length:
```
‖e_1‖² = 0.9239² + 0.3827² + (-0.3827)² + (-0.9239)²
       ≈ 0.8536 + 0.1464 + 0.1464 + 0.8536 = 2.0
‖e_1‖  = sqrt(2) = sqrt(N/2)      (N=4, so N/2=2, sqrt(2) ≈ 1.4142)
```
(k=2 and k=3 also come out with length sqrt(2) — every cosine basis vector with k>0 has the same length. That's a necessary consequence of the orthogonality of trig functions, not a coincidence.)

**Here comes the key step**: we want every basis vector, once multiplied by its weight, to have length 1 ("unit length" — that's exactly what "normalization" means). Since we've already computed the lengths, the weight is simply **the reciprocal of the length**:
```
weight for k=0 = 1 / ‖e_0‖ = 1 / 2 = 1 / sqrt(4) = sqrt(1/N)     ✓ matches the lesson's formula w[0]=sqrt(1/N)
weight for k>0 = 1 / ‖e_1‖ = 1 / sqrt(2) = sqrt(1/2) = sqrt(2/4) = sqrt(2/N)   ✓ matches the lesson's formula w[k>0]=sqrt(2/N)
```

Now we can answer head-on the question "if I derive a new transform, how should I set the weights?": **the weights are never numbers the designer picks at will; they are "1 divided by that basis vector's natural length".** The all-ones vector is inherently "longer" than the cosine vectors (sqrt(N) is bigger than sqrt(N/2)), so pulling them all back to unit length naturally needs different scale factors — one rule (divide by your own length) applied to vectors of two different lengths automatically gives two different answers, not two independently designed rules.

The code below checks both N=4 and this lesson's actual N=40 (n_mels), confirming that the derivation matches the weights in Aurora's code exactly:


In [ ]:
# Check: the L2 norms of the DCT-II basis vectors — why k=0 and k>0 get different weights
import numpy as np

def dct_basis_and_norm(N, k):
    n = np.arange(N)
    e_k = np.cos(np.pi / N * (n + 0.5) * k)
    return e_k, np.linalg.norm(e_k)

for N in [4, 40]:
    e0, norm0 = dct_basis_and_norm(N, 0)
    e1, norm1 = dct_basis_and_norm(N, 1)
    print(f"N={N}:")
    print(f"  k=0 basis vector: {e0[:4].round(4)}{' ...' if N > 4 else ''}  → ‖e_0‖ = {norm0:.4f}  (theory: sqrt(N) = {np.sqrt(N):.4f})")
    print(f"  k=1 basis vector: {e1[:4].round(4)}{' ...' if N > 4 else ''}  → ‖e_1‖ = {norm1:.4f}  (theory: sqrt(N/2) = {np.sqrt(N / 2):.4f})")
    print(f"  → weight w[0]   = 1/‖e_0‖ = {1 / norm0:.4f}   ,  sqrt(1/N) = {np.sqrt(1 / N):.4f}")
    print(f"  → weight w[k>0] = 1/‖e_1‖ = {1 / norm1:.4f}   ,  sqrt(2/N) = {np.sqrt(2 / N):.4f}\n")

print("💡 Whatever N is, the k=0 basis vector has norm sqrt(N) and every k>0 basis vector has norm sqrt(N/2),")
print("   so the weight formulas w[0]=sqrt(1/N), w[k>0]=sqrt(2/N) hold for any N (including this lesson's n_mels=40).")

In [ ]:
# Demo: how coefficient energy is distributed for different n_mfcc
from aurora.audio.mfcc import dct_ii
from aurora.audio.mel import mel_spectrogram, power_to_db

sr = 16000
x = sine(440, duration=1.0, sample_rate=sr)
log_mel = power_to_db(mel_spectrogram(x, sr, n_mels=40, n_fft=1024, hop_length=256), top_db=None)
all_coeffs = dct_ii(log_mel)  # (n_frames, 40)

energy_per_coeff = np.mean(all_coeffs**2, axis=0)  # mean square of each coefficient over all frames

plt.figure(figsize=(8, 3))
plt.bar(range(len(energy_per_coeff)), energy_per_coeff, color="steelblue")
plt.axvspan(-0.5, 12.5, color="green", alpha=0.15, label="keep k=0..12 (n_mfcc=13, includes k=0)")
plt.xlabel("DCT coefficient index k")
plt.ylabel("mean energy (mean square)")
plt.title("Mean energy of each DCT coefficient — concentrated in the low orders")
plt.legend()
plt.tight_layout()
plt.show()

## 3. The complete flow chart

```
raw audio x
  │
  ▼ STFT (Hann window, win_len=1024, hop=256)
power spectrum |X[k]|²   shape: (n_frames, n_fft//2+1)
  │
  ▼ mel_filterbank @   [mel matrix, shape: (n_mels, n_fft//2+1)]
mel energies       shape: (n_frames, n_mels)
  │
  ▼ log (power_to_db, top_db=None)
log-mel spectrum   shape: (n_frames, n_mels)
  │
  ▼ DCT-II (ortho normalization)
DCT coefficients   shape: (n_frames, n_mels)
  │
  ▼ take [:, :n_mfcc]    k=0..n_mfcc-1
MFCC               shape: (n_frames, n_mfcc)
```

Every step is a linear transform or an elementwise operation, with no learnable parameters. The whole pipeline has only four hyperparameters: `n_fft`, `hop`, `n_mels`, `n_mfcc`.

### The matrix dimensions, broken down frame by frame

**The mel-filtering step is where dimensions most often get mixed up.** Let's clear it up with concrete numbers:

```
Suppose: n_frames = 100, n_fft = 1024, n_mels = 40, sr = 16000

power has shape: (100, 513)        fb has shape: (40, 513)
  ↓ each row is one frame's            ↓ each row is one mel filter
    power spectrum

  frame 0: [P₀₀, P₀₁, ..., P₀₅₁₂]
  frame 1: [P₁₀, P₁₁, ..., P₁₅₁₂]     mel 0: [M₀₀, M₀₁, ..., M₀₅₁₂]
  frame 2: [P₂₀, P₂₁, ..., P₂₅₁₂]     mel 1: [M₁₀, M₁₁, ..., M₁₅₁₂]
  ...                                 ...
  frame99: [P₉₉₀, P₉₉₁, ..., P₉₉₅₁₂]  mel39: [M₃₉₀, M₃₉₁, ..., M₃₉₅₁₂]
```

We want: **the j-th mel energy of frame i** = frame i's power spectrum · the j-th mel filter (a dot product)

```
Mel[i, j] = power[i, :] · fb[j, :]
          = sum_k( power[i, k] * fb[j, k] )
```

As a matrix product:
```
Mel[i, j] = (power @ fb.T)[i, j]
          = (100, 513) @ (513, 40)  ← dimension check: the inner 513s cancel, giving (100, 40) ✓
```

The resulting Mel matrix is (n_frames, n_mels); each row is one frame's 40 mel energies.

In [ ]:
# Demo: walk through the pipeline step by step, printing each step's shape and values
from aurora.audio.stft import power_spectrogram
from aurora.audio.mel import mel_filterbank, power_to_db
from aurora.audio.mfcc import dct_ii

sr = 16000
n_fft, hop, n_mels, n_mfcc = 1024, 256, 40, 13
x = sine(440, duration=1.0, sample_rate=sr)
print(f"x.shape           = {x.shape}")
print(f"✓ raw signal: 1 second @ 16 kHz = 16000 samples\n")

power = power_spectrogram(x, n_fft=n_fft, hop_length=hop)
print(f"power.shape       = {power.shape}  ← (n_frames, n_fft//2+1)")
print(f"  - n_frames = 1 + {len(x)} // {hop} = {power.shape[0]}")
print(f"  - frequency bins = {n_fft}//2 + 1 = {power.shape[1]}")
print(f"✓ power spectrum: 63 frames, 513 frequency components per frame\n")

fb = mel_filterbank(n_mels, n_fft, sr)
print(f"mel_filterbank shape = {fb.shape}  ← (n_mels, n_fft//2+1)")
print(f"✓ 40 mel filters, each acting on 513 frequencies\n")

mel = power @ fb.T
print(f"mel.shape         = {mel.shape}  ← (n_frames, n_mels)")
print(f"✓ the first 10 mel energies of frame 0: {mel[0, :10].round(2)}")
print(f"✓ energy range after mel filtering: min={mel.min():.3e}, max={mel.max():.3e}\n")

log_mel = power_to_db(mel, top_db=None)
print(f"log_mel.shape     = {log_mel.shape}")
print(f"✓ after the log (10·log10) the dynamic range is heavily compressed")
print(f"✓ the first 10 log-mel values of frame 0: {log_mel[0, :10].round(2)} dB")
print(f"✓ value range: min={log_mel.min():.1f}, max={log_mel.max():.1f} dB\n")

coeffs = dct_ii(log_mel)
print(f"dct_coeffs.shape  = {coeffs.shape}")
print(f"✓ the DCT-II transforms along the last axis of each frame (each frame handled independently)")
print(f"✓ DCT coefficients k=0..5 of frame 0: {coeffs[0, :6].round(3)}")
print(f"  → k=0 (DC) is the largest; by k=5 it has decayed to near 0\n")

mfcc_out = coeffs[:, :n_mfcc]
print(f"mfcc_out.shape    = {mfcc_out.shape}  ← (n_frames, n_mfcc) final output")
print(f"✓ keep the first 13 DCT coefficients as the MFCC features")
print(f"✓ final compression: 16000 samples → {mfcc_out.size} MFCC values")

## 4. ✏️ Implement `my_mfcc(x, sr, n_mfcc=13, n_mels=40, win_len=1024, hop=256)`

**Four steps in a row**:

| Step | Code | Output shape |
|---|---|---|
| 1 | `S = mel_spectrogram(x, sr, n_fft=win_len, ...)` | `(n_frames, n_mels)` — power mel spectrum |
| 2 | `logS = power_to_db(S, top_db=None)` | `(n_frames, n_mels)` — 10·log10, with the built-in 1e-10 floor |
| 3 | `coeffs = dct_ii(logS, norm="ortho")` | `(n_frames, n_mels)` — transforms frame by frame along the last axis |
| 4 | `return coeffs[:, :n_mfcc]` | `(n_frames, n_mfcc)` |

**Acceptance criteria**:
- `np.allclose(my_mfcc(x, sr), aurora_mfcc(x, sr), atol=1e-5)` passes
- output shape `(n_frames, n_mfcc)` = `(n_frames, 13)`
- no nan/inf (the `amin=1e-10` floor in `power_to_db` does its job)

> Note: the log step must use `power_to_db` (10·log10), not `np.log` — the two differ by a factor of 10/ln10 ≈ 4.34, and using the wrong one means your numbers won't match `aurora_mfcc`.
> Available imports: `from aurora.audio.mel import mel_spectrogram, power_to_db`, `from aurora.audio.mfcc import dct_ii`

### ⚠️ Key point: how dct_ii handles a 2-D matrix

Here's something that's easy to misunderstand. `log_mel` is an `(n_frames, n_mels)` matrix, and when you call `dct_ii(log_mel)`:

it does **not** do one 2-D DCT over the whole matrix (that would give a completely different result); it does **a separate 1-D DCT on each row (each frame)**.

```
log_mel shape: (100, 40)            dct_ii(log_mel) shape: (100, 40)

log_mel[0, :] = [x₀, x₁, ..., x₃₉]  → dct_ii(log_mel[0, :]) = [y₀, y₁, ..., y₃₉]
log_mel[1, :] = [x₀, x₁, ..., x₃₉]  → dct_ii(log_mel[1, :]) = [y₀, y₁, ..., y₃₉]
...
log_mel[99, :] = [...]               → ...
```

Aurora's `dct_ii` already handles this "row-by-row transform" internally, so you **don't need to write a loop**. Pass in the 2-D matrix directly and it applies the DCT to each row along the last axis.

This is different from the STFT and the mel filtering — those are "matrix products" that process all frames at once; the DCT is a "frame-independent" transform.

## 4a. A 20-second self-check before diving into L50

These aren't new ideas; they just spell out what this lesson depends on from earlier lessons.
Mark each one `True/False` first; if you're unsure, go back to the matching lesson.

In [ ]:
pipeline_checklist = {
    "stft_shape_ok": None,  # stuck? back to L44: first confirm the STFT output is (T, n_bins)
    "mel_matmul_ok": None,  # stuck? back to L47: first confirm the direction of power @ fb.T
    "log_floor_ok": None,   # stuck? back to L47: think through why log(max(power, eps)) needs eps
    "dct_ortho_ok": None,   # stuck? back to L49: review the DCT-II's orthonormalization
}

pending = [name for name, value in pipeline_checklist.items() if value is None]
print("Change None to True/False; if you're unsure, review the matching lesson first.")
print("Still to fill in:", pending)

In [ ]:
def my_mfcc(x, sr, n_mfcc=13, n_mels=40, win_len=1024, hop=256):
    """MFCC extraction: x -> (n_frames, n_mfcc)"""
    from aurora.audio.mel import mel_spectrogram, power_to_db
    from aurora.audio.mfcc import dct_ii

    # ✏️ TODO 1: compute the mel power spectrum, shape (n_frames, n_mels)
    raise NotImplementedError("TODO: mel_spectrogram(x, sr, n_mels=n_mels, n_fft=win_len, hop_length=hop)")

    # ✏️ TODO 2: take the log (power_to_db, top_db=None); shape unchanged
    raise NotImplementedError("TODO: power_to_db(mel, top_db=None)")

    # ✏️ TODO 3: DCT-II each frame (norm="ortho"), shape (n_frames, n_mels)
    raise NotImplementedError("TODO: dct_ii applies the DCT-II to each frame log_mel[i]")

    # ✏️ TODO 4: keep the first n_mfcc coefficients: [:, :n_mfcc], shape (n_frames, n_mfcc)
    raise NotImplementedError("TODO: full MFCC pipeline: STFT→Mel→log→DCT")

In [ ]:
from aurora.audio.mfcc import mfcc as aurora_mfcc

sr = 16000
x = sine(440, duration=1.0, sample_rate=sr)

try:
    out = my_mfcc(x, sr, n_mfcc=13, n_mels=40, win_len=1024, hop=256)
except (NotImplementedError, TypeError):
    out = None

if out is None:
    print('⬜ my_mfcc is not implemented yet; finish the four TODOs and run again')
else:
    ref = aurora_mfcc(x, sr, n_mfcc=13, n_fft=1024, hop_length=256, n_mels=40)
    print(f'my_mfcc shape : {out.shape}')
    print(f'aurora  shape : {ref.shape}')
    assert out.shape == ref.shape, f'shape mismatch: {out.shape} vs {ref.shape}'
    assert np.allclose(out, ref, atol=1e-5), 'values differ; check the TODO steps'
    print('✅ shape and values both match aurora.audio.mfcc.mfcc()')


## 5. Parameter experiment: an MFCC heatmap

**Goal of the experiment**: visualize how the 13 coefficients change over time.

- **x-axis**: frame number (time)
- **y-axis**: coefficient index k = 0–12 (lower → coarser spectral shape)
- **What to expect**:
  - the lowest coefficients change relatively smoothly; they represent the overall timbre envelope
  - the top coefficients (k ≈ 11–12) change fastest and carry fine-grained spectral texture
  - for a pure sine wave the high-order coefficients are close to zero (the spectrum is very clean); try switching to a `chirp` signal and look at the difference

Change to `n_mfcc=20` and the y-axis extends, so you can see the energy die away quickly in the higher orders.

In [ ]:
from aurora.audio.io import chirp

sr = 16000
x_chirp = chirp(f0=200, f1=4000, duration=2.0, sample_rate=sr)

try:
    mfcc_mat = my_mfcc(x_chirp, sr, n_mfcc=13, n_mels=40, win_len=1024, hop=256)
except (NotImplementedError, TypeError):
    mfcc_mat = None

if mfcc_mat is None:
    print('⬜ finish the my_mfcc TODO above first, then run this cell')
else:
    import matplotlib.pyplot as plt, numpy as np
    plt.figure(figsize=(10, 4))
    plt.imshow(
        mfcc_mat.T,
        aspect='auto', origin='lower', cmap='magma',
    )
    plt.colorbar(label='MFCC value')
    plt.xlabel('frame number'); plt.ylabel('coefficient index k')
    plt.title('MFCC heatmap of a chirp (frequency sweeping 200→4000 Hz)')
    plt.tight_layout(); plt.show()
    print(f'✅ mfcc_mat.shape = {mfcc_mat.shape}')


## 🎯 The payoff later
Today you folded STFT → mel → log → DCT into a single `my_mfcc()`. These 13-dimensional features come back in **L51 (real WAV files) / L62 (the keyword-spotting dataset) / L70 (acoustic features for ASR)** — and there they're no longer an exercise but the **real input** fed to a neural network. Every shape you got right by hand today is what lets the later models run at all.

## Wrap-up

`my_mfcc()` chains four steps, `mel_spectrogram → power_to_db → dct_ii → [:n_mfcc]`, and outputs a float64 array of shape `(n_frames, n_mfcc)` whose values match `aurora.audio.mfcc.mfcc()` exactly (atol=1e-5). This function is the core of the `aurora.audio.mfcc` module; the Month 2 keyword-spotting classifier takes the `(n_frames, 13)` MFCC matrix directly as its input features. The next lesson (L51) moves this pipeline onto real (or synthetic) WAV audio: reading the file, four layers of visualization (waveform / spectrogram / mel / MFCC), and optionally checking answers against librosa.

## ✏️ Closed-book derivation check cell — tracking dimensions through the MFCC pipeline

**Rule: collapse every cell above and do the derivation below from memory.**

**Problem**: given the parameters:
- signal length `L = 16000` (1 second at 16 kHz)
- `n_fft = 1024`, `hop = 256`, `n_mels = 40`, `n_mfcc = 13`
- aurora's `stft` defaults to `center=True` (reflect-padding `n_fft//2` at each end), so `n_frames = 1 + L // hop`

Write down the output shape of each step:

| Step | Operation | Output shape |
|------|------|---------|
| 1 | signal | `(L,)` = (___,) |
| 2 | STFT → magnitude | (___,  ___) |
| 3 | mel filtering | (___, ___) |
| 4 | log (dB) | (___, ___) |
| 5 | DCT-II → MFCC | (___, ___) |

(Fill in the table here...)

In [ ]:
# Check: each step's output shape matches the derivation
import numpy as np, sys
sys.path.insert(0, 'src')
from aurora.audio.stft import stft
from aurora.audio.mel import mel_filterbank, power_to_db
from aurora.audio.mfcc import mfcc, dct_ii

SR, N_FFT, HOP, N_MELS, N_MFCC = 16000, 1024, 256, 40, 13
signal = np.sin(2*np.pi*440*np.arange(SR)/SR).astype(np.float32)

spec   = stft(signal, n_fft=N_FFT, hop_length=HOP)      # (n_frames, n_bins)
n_frames, n_bins = spec.shape
fb     = mel_filterbank(n_mels=N_MELS, n_fft=N_FFT, sample_rate=SR)
mel_e  = (np.abs(spec)**2) @ fb.T                        # (n_frames, N_MELS)
log_m  = power_to_db(mel_e)                              # (n_frames, N_MELS)
coeffs = mfcc(signal, sample_rate=SR, n_mfcc=N_MFCC,
              n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS) # (n_frames, N_MFCC)

shapes = [spec.shape, mel_e.shape, log_m.shape, coeffs.shape]
labels = ['STFT', 'Mel', 'log-Mel', 'MFCC']
for l, s in zip(labels, shapes): print(f'  {l}: {s}')
assert coeffs.shape == (n_frames, N_MFCC), f'final shape {coeffs.shape} is not what we expected'
print('✅ every dimension in the MFCC pipeline checks out')

In [ ]:
# ✏️ Self-review for this lesson
l50_review = {
    "pipeline_5steps":         None,  # remember the five steps STFT→Mel→log→DCT→truncate? True/False
    "my_mfcc_implemented":     None,  # my_mfcc implemented and passing the atol=1e-5 comparison? True/False
    "shape_tracking":          None,  # can work out each step's output shape by hand? True/False
    "n_mfcc_13_reason":        None,  # understand first 13 dims = vocal-tract shape (vowels), higher dims = noise? True/False
    "whiteboard_passed":       None,  # passed the closed-book whiteboard challenge on dimensions? True/False
}

unfilled = [k for k, v in l50_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l50_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L50 fully cleared! On to L51: MFCCs on real audio')

---

→ **Next lesson** · [L51 · MFCCs on real audio](L51_real_audio.ipynb)

> Next lesson covers **MFCCs on real audio**: extracting features from real WAV audio, with librosa as the answer key.